### Part A: Data Understanding and Regex

#### Task 1: Load and Explore

In [53]:
import re, pandas as pd
df = pd.read_csv('nlp_customer_messages - nlp_customer_messages.csv')
print(df.head(5))

print("\n--------------------------------------------------------\n")

print(df.shape, df.columns.tolist() , end="\n\n")
print(df.isna().sum())

df['char_count'] = df['message'].str.len()

  message_id                                            message
0     MSG001  Good morning, My phone is not working :(  Than...
1     MSG002  Hi, Thanks for your quick response! I need hel...
2     MSG003  Hi support, I LOVE this product 😍😍 Can someone...
3     MSG004  Hey there, The new update is AMAZING!! Please ...
4     MSG005  Dear team, The price is 250 EGP, too expensive...

--------------------------------------------------------

(80, 2) ['message_id', 'message']

message_id    0
message       0
dtype: int64


#### Task 2: Regex Extract

In [54]:
EMAIL  = r'[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+'
URL    = r'(?:https?://|www\.)[^\s]*[^\s.,!?;:)]'
PHONE  = r'(?<!\d)010\d{8}(?!\d)'

def extract_all(t):
    return {
        'emails':  re.findall(EMAIL, t),
        'urls':    re.findall(URL, t),
        'phones':  re.findall(PHONE, t),
        'numbers': re.findall(r'\d+', t),
        'has_urgent': bool(re.search(r'\burgent\b', t, re.IGNORECASE)),
    }

#### Task 3: Regex Clean

In [55]:
def clean_text(t):
    t = re.sub(EMAIL, '[EMAIL]', t)
    t = re.sub(URL, '[URL]', t)
    t = re.sub(PHONE, '[PHONE]', t)
    t = re.sub(r'\d+', '[NUMBER]', t)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()
df['clean_message'] = df['message'].apply(clean_text)

#### Task 4: Test Your Regex

In [56]:
tests = [
    "Contact me at sara.ali@mail.com please",          # email
    "My number is 01234567890 or 01099887766 now",     # phone
    "Check www.shop.com/item?id=5, it's URGENT.",      # url
    "I am happy with the service.",                    # no email/url/phone
]
for s in tests:
    print("TEXT   :", s)
    print("extract:", extract_all(s))
    print("clean  :", clean_text(s), "\n")

TEXT   : Contact me at sara.ali@mail.com please
extract: {'emails': ['sara.ali@mail.com'], 'urls': [], 'phones': [], 'numbers': [], 'has_urgent': False}
clean  : Contact me at [EMAIL] please 

TEXT   : My number is 01234567890 or 01099887766 now
extract: {'emails': [], 'urls': [], 'phones': ['01099887766'], 'numbers': ['01234567890', '01099887766'], 'has_urgent': False}
clean  : My number is [NUMBER] or [PHONE] now 

TEXT   : Check www.shop.com/item?id=5, it's URGENT.
extract: {'emails': [], 'urls': ['www.shop.com/item?id=5'], 'phones': [], 'numbers': ['5'], 'has_urgent': True}
clean  : Check [URL], it's URGENT. 

TEXT   : I am happy with the service.
extract: {'emails': [], 'urls': [], 'phones': [], 'numbers': [], 'has_urgent': False}
clean  : I am happy with the service. 



### Part B: Tokenization

#### Task 5: NLTK

In [57]:
import nltk
nltk.download('punkt'); nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize

df['sentences_nltk'] = df['clean_message'].apply(sent_tokenize)
df['tokens_nltk']    = df['clean_message'].apply(word_tokenize)
df['sentence_count'] = df['sentences_nltk'].apply(len)
df['token_count']    = df['tokens_nltk'].apply(len)

print(df.nlargest(5, 'token_count')[['message_id', 'token_count', 'clean_message']])

   message_id  token_count                                      clean_message
65     MSG066           38  Hi, The package for order [NUMBER] is damaged....
35     MSG036           37  Hey there, I need help with payment for order ...
71     MSG072           36  Hello team, I cannot log in to my account. Ple...
44     MSG045           35  Dear team, The price is [NUMBER] EGP, too expe...
59     MSG060           35  Hey there, My order [NUMBER] arrived late!!! P...


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


#### Task 6: spaCy

In [58]:
import spacy
nlp = spacy.load('en_core_web_sm')

first10 = df['clean_message'].head(10)
for mid, text in zip(df['message_id'].head(10), first10):
    sp  = [t.text for t in nlp(text)]
    nl  = word_tokenize(text)
    if sp != nl:                 
        print(mid); print("NLTK :", nl); print("spaCy:", sp)
        break
else:
    print("No difference in the first 10 messages")

doc = nlp(df['clean_message'].iloc[2])
for t in doc:
    print(f"{t.text:15} alpha={t.is_alpha!s:6} punct={t.is_punct}")

MSG001
NLTK : ['Good', 'morning', ',', 'My', 'phone', 'is', 'not', 'working', ':', '(', 'Thanks', 'again', '!']
spaCy: ['Good', 'morning', ',', 'My', 'phone', 'is', 'not', 'working', ':(', 'Thanks', 'again', '!']
Hi              alpha=True   punct=False
support         alpha=True   punct=False
,               alpha=False  punct=True
I               alpha=True   punct=False
LOVE            alpha=True   punct=False
this            alpha=True   punct=False
product         alpha=True   punct=False
😍               alpha=False  punct=False
😍               alpha=False  punct=False
Can             alpha=True   punct=False
someone         alpha=True   punct=False
help            alpha=True   punct=False
me              alpha=True   punct=False
?               alpha=False  punct=True
Email           alpha=True   punct=False
:               alpha=False  punct=True
[               alpha=False  punct=True
EMAIL           alpha=True   punct=False
]               alpha=False  punct=True


### Part C: Case Folding

In [59]:
import re, nltk, spacy
from itertools import chain
from collections import Counter
import pandas as pd
for p in ['punkt','punkt_tab','wordnet','omw-1.4','stopwords','averaged_perceptron_tagger_eng']:
    nltk.download(p)
from nltk.tokenize import word_tokenize
nlp = spacy.load('en_core_web_sm')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\CM\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


#### Task 7 — Lowercase and Vocabulary Size

In [60]:
df['lower_message'] = df['clean_message'].str.lower()
df['lower_tokens']  = df['lower_message'].apply(word_tokenize)

before = set(chain.from_iterable(df['tokens_nltk'])) 
after  = set(chain.from_iterable(df['lower_tokens']))
print("Unique tokens before lowercase:", len(before))
print("Unique tokens after  lowercase:", len(after))
print("Reduction:", len(before) - len(after))

groups = {}
for tok in before:
    groups.setdefault(tok.lower(), set()).add(tok)
examples = {k: v for k, v in groups.items() if len(v) > 1}
print(examples)
print(df[df['clean_message'].str.contains(r'\bNOT\b')][['message_id','clean_message']].head(3))

Unique tokens before lowercase: 122
Unique tokens after  lowercase: 113
Reduction: 9
{'this': {'This', 'this'}, 'email': {'Email', 'EMAIL', 'email'}, 'great': {'great', 'Great'}, 'my': {'my', 'My'}, 'phone': {'PHONE', 'phone'}, 'the': {'The', 'the'}, 'can': {'can', 'Can'}, 'not': {'not', 'NOT'}}
   message_id                                      clean_message
16     MSG017  Good morning, I did NOT receive my package! Th...
36     MSG037  Dear team, I did NOT receive my package! Thank...
56     MSG057  Good morning, I did NOT receive my package! Th...


### Part D — Stemming and Lemmatization


#### Task 8 — Stemming with NLTK


In [61]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer
porter, snowball, lancaster = PorterStemmer(), SnowballStemmer('english'), LancasterStemmer()

df['alpha_tokens']   = df['lower_tokens'].apply(lambda ts: [t for t in ts if t.isalpha()])
df['porter_stems']   = df['alpha_tokens'].apply(lambda ts: [porter.stem(t)   for t in ts])
df['snowball_stems'] = df['alpha_tokens'].apply(lambda ts: [snowball.stem(t) for t in ts])
df['lancaster_stems']= df['alpha_tokens'].apply(lambda ts: [lancaster.stem(t) for t in ts])

words = ['delivery','received','crashes','working','disappointed',
        'expensive','quality','arrived','waiting','ordered']
stem_table = pd.DataFrame({
    'original': words,
    'porter':   [porter.stem(w)   for w in words],
    'snowball': [snowball.stem(w) for w in words],
    'lancaster':[lancaster.stem(w) for w in words],
})
print(stem_table)

       original      porter    snowball   lancaster
0      delivery    deliveri    deliveri    delivery
1      received      receiv      receiv      receiv
2       crashes       crash       crash       crash
3       working        work        work        work
4  disappointed  disappoint  disappoint  disappoint
5     expensive      expens      expens      expend
6       quality     qualiti     qualiti        qual
7       arrived       arriv       arriv          ar
8       waiting        wait        wait        wait
9       ordered       order       order         ord


#### Task 9 — Lemmatization with NLTK and spaCy


In [62]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()
df['lemmas_nltk']  = df['alpha_tokens'].apply(lambda ts: [wnl.lemmatize(t) for t in ts])
df['lemmas_spacy'] = df['lower_message'].apply(lambda t: [tok.lemma_ for tok in nlp(t) if tok.is_alpha])

spacy_map = {}
for doc in nlp.pipe(df['lower_message']):
    for tok in doc:
        if tok.is_alpha: spacy_map.setdefault(tok.text, tok.lemma_)

lem_table = pd.DataFrame({
    'original':    words,
    'stem_porter': [porter.stem(w) for w in words],
    'lemma_nltk':  [wnl.lemmatize(w) for w in words],
    'lemma_spacy': [spacy_map.get(w, '?') for w in words],
})
print(lem_table)
print(lem_table[lem_table.stem_porter != lem_table.lemma_spacy])  

from nltk.corpus import wordnet as wn
POSMAP = {'NOUN': wn.NOUN, 'VERB': wn.VERB, 'ADJ': wn.ADJ, 'ADV': wn.ADV}
def lemma_pos_aware(text):
    return [wnl.lemmatize(t.text, POSMAP.get(t.pos_, wn.NOUN)) for t in nlp(text) if t.is_alpha]
df['lemmas_nltk_pos'] = df['lower_message'].apply(lemma_pos_aware)

       original stem_porter    lemma_nltk   lemma_spacy
0      delivery    deliveri      delivery      delivery
1      received      receiv      received       receive
2       crashes       crash         crash         crash
3       working        work       working          work
4  disappointed  disappoint  disappointed  disappointed
5     expensive      expens     expensive     expensive
6       quality     qualiti       quality       quality
7       arrived       arriv       arrived        arrive
8       waiting        wait       waiting          wait
9       ordered       order       ordered         order
       original stem_porter    lemma_nltk   lemma_spacy
0      delivery    deliveri      delivery      delivery
1      received      receiv      received       receive
4  disappointed  disappoint  disappointed  disappointed
5     expensive      expens     expensive     expensive
6       quality     qualiti       quality       quality
7       arrived       arriv       arrived       

### Part E — Stop Words


#### Task 10 — Remove Stop Words (Two Libraries)


In [63]:
from nltk.corpus import stopwords
nltk_sw  = set(stopwords.words('english'))
spacy_sw = set(nlp.Defaults.stop_words)

df['tokens_no_stop_nltk']  = df['lower_tokens'].apply(lambda ts: [t for t in ts if t not in nltk_sw])
df['tokens_no_stop_spacy'] = df['lower_tokens'].apply(lambda ts: [t for t in ts if t not in spacy_sw])

cmp = pd.DataFrame({
    'message_id': df['message_id'].head(10),
    'before':     df['lower_tokens'].head(10).apply(len),
    'after_nltk': df['tokens_no_stop_nltk'].head(10).apply(len),
    'after_spacy':df['tokens_no_stop_spacy'].head(10).apply(len),
})
print(cmp)

print(len(nltk_sw), len(spacy_sw), nltk_sw == spacy_sw)
print("only NLTK :", sorted(nltk_sw - spacy_sw)[:20])
print("only spaCy:", sorted(spacy_sw - nltk_sw)[:20])

negs = {'not', 'no', 'never'}
for w in negs:
    print(w, "| in NLTK list:", w in nltk_sw, "| in spaCy list:", w in spacy_sw)
has_neg = df[df['lower_tokens'].apply(lambda ts: bool(negs & set(ts)))]
print(has_neg[['message_id','lower_message']].head(5))

nltk_sw_keep  = nltk_sw  - negs
spacy_sw_keep = spacy_sw - negs
df['tokens_keep_neg_nltk']  = df['lower_tokens'].apply(lambda ts: [t for t in ts if t not in nltk_sw_keep])
df['tokens_keep_neg_spacy'] = df['lower_tokens'].apply(lambda ts: [t for t in ts if t not in spacy_sw_keep])

i = has_neg.index[0]
print("Original      :", df.loc[i, 'lower_tokens'])
print("Default NLTK  :", df.loc[i, 'tokens_no_stop_nltk'])
print("Keep negation :", df.loc[i, 'tokens_keep_neg_nltk'])

  message_id  before  after_nltk  after_spacy
0     MSG001      13           9            9
1     MSG002      15          12           12
2     MSG003      18          14           13
3     MSG004      17          14           13
4     MSG005      23          17           16
5     MSG006      27          21           20
6     MSG007      17          12           12
7     MSG008      22          21           20
8     MSG009      24          21           21
9     MSG010      22          16           15
198 326 False
only NLTK : ['ain', 'aren', "aren't", 'couldn', "couldn't", 'd', 'didn', "didn't", 'doesn', "doesn't", 'don', "don't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'having', "he'd"]
only spaCy: ["'d", "'ll", "'m", "'re", "'s", "'ve", 'across', 'afterwards', 'almost', 'alone', 'along', 'already', 'also', 'although', 'always', 'among', 'amongst', 'amount', 'another', 'anyhow']
never | in NLTK list: False | in spaCy list: True
not | in NLTK list: True | in spaCy list:

### Part F — Linguistic Analysis

#### Task 11 — Part-of-Speech (POS) Tagging


In [64]:
docs = list(nlp.pipe(df['clean_message']))

for mid, doc in list(zip(df['message_id'], docs))[:5]:
    print(mid)
    for t in doc:
        print(f"  {t.text:15} {t.pos_:6} {t.tag_}")

pos_counts = Counter(t.pos_ for d in docs for t in d)
print(pd.DataFrame(pos_counts.most_common(), columns=['POS','count']))

verbs = list(dict.fromkeys(t.lemma_.lower() for d in docs for t in d if t.pos_ == 'VERB'))[:5]
adjs  = list(dict.fromkeys(t.text.lower()  for d in docs for t in d if t.pos_ == 'ADJ'))[:5]
print("Verbs:", verbs); print("Adjectives:", adjs)

# اختياري: NLTK
print(nltk.pos_tag(word_tokenize(df['clean_message'].iloc[0])))

MSG001
  Good            ADJ    JJ
  morning         NOUN   NN
  ,               PUNCT  ,
  My              PRON   PRP$
  phone           NOUN   NN
  is              AUX    VBZ
  not             PART   RB
  working         VERB   VBG
  :(              PUNCT  NFP
  Thanks          NOUN   NNS
  again           ADV    RB
  !               PUNCT  .
MSG002
  Hi              INTJ   UH
  ,               PUNCT  ,
  Thanks          NOUN   NNS
  for             ADP    IN
  your            PRON   PRP$
  quick           ADJ    JJ
  response        NOUN   NN
  !               PUNCT  .
  I               PRON   PRP
  need            VERB   VBP
  help            NOUN   NN
  ASAP            ADV    RB
  !               PUNCT  .
  !               PUNCT  .
  !               PUNCT  .
MSG003
  Hi              INTJ   UH
  support         NOUN   NN
  ,               PUNCT  ,
  I               PRON   PRP
  LOVE            VERB   VBP
  this            DET    DT
  product         NOUN   NN
  😍               NOUN

#### Task 12 — Named Entity Recognition (NER)


In [65]:
docs_raw = list(nlp.pipe(df['message']))  

for mid, doc in list(zip(df['message_id'], docs_raw))[:15]:
    print(mid, [(e.text, e.label_) for e in doc.ents] or "no entities")

rows = [(mid, e.text, e.label_) for mid, doc in zip(df['message_id'], docs_raw) for e in doc.ents]
entities_df = pd.DataFrame(rows, columns=['message_id', 'entity_text', 'entity_label'])
print(entities_df['entity_label'].value_counts())

for i in [2, 4, 5, 8, 19]:
    print(df.loc[i,'message_id'], '|', df.loc[i,'message'])
    print("   ents :", [(e.text, e.label_) for e in docs_raw[i].ents])
    print("   regex:", extract_all(df.loc[i,'message']))

MSG001 no entities
MSG002 no entities
MSG003 no entities
MSG004 no entities
MSG005 [('250', 'CARDINAL'), ('EGP', 'ORG'), ('01010011705', 'DATE')]
MSG006 [('10102', 'DATE')]
MSG007 [('@support', 'GPE')]
MSG008 [('7 days', 'DATE')]
MSG009 no entities
MSG010 [('01010023410', 'DATE')]
MSG011 [('A-45-2026', 'DATE')]
MSG012 no entities
MSG013 [('2', 'CARDINAL'), ('1', 'CARDINAL')]
MSG014 [('#', 'CARDINAL'), ('@support', 'GPE')]
MSG015 [('01010035115', 'DATE')]
entity_label
DATE        43
CARDINAL    19
GPE          9
ORG          6
PERSON       6
Name: count, dtype: int64
MSG003 | Hi support, I LOVE this product 😍😍 Can someone help me? Email: customer3@example.com
   ents : []
   regex: {'emails': ['customer3@example.com'], 'urls': [], 'phones': [], 'numbers': ['3'], 'has_urgent': False}
MSG005 | Dear team, The price is 250 EGP, too expensive. Thank you. Call me at 01010011705
   ents : [('250', 'CARDINAL'), ('EGP', 'ORG'), ('01010011705', 'DATE')]
   regex: {'emails': [], 'urls': [], 'phone